<a href="https://colab.research.google.com/github/godwinbaah/DEVELOPING-A-CODEBERT-BASED-MODEL-FOR-SQL-INJECTION-DETECTION-IN-PYTHON-CODE/blob/main/SQLi_CodeBERT_Colab_Experiment_v3.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# CodeBERT-Based SQL Injection Detection — Experiment Notebook (v3)

**What changed in this version, and why:**

| Original plan | This version | Reason |
|---|---|---|
| Juliet + CVEfixes + Big-Vul | **CVEfixes only** | Juliet has no native Python release (C/C++/Java/C# only). Big-Vul is confirmed C/C++-only by its own paper (Fan et al., 2020) — it has no Python samples to filter to, at all. Both are excluded from the trained/evaluated dataset; Juliet remains referenced in the thesis as background/taxonomy literature only. |
| SonarQube + Bandit + Semgrep | **Bandit + Semgrep only** | SonarQube requires a persistent server process incompatible with the free-tier, session-limited Colab environment. Documented as a disclosed scope adjustment in Chapter 3. |

This notebook implements the pipeline described in the (now updated) Chapter Three and Chapter Four:

1. Environment setup and Google Drive checkpointing
2. Dataset construction from **CVEfixes**, with a schema-detecting loader (see Section 2 — your actual downloaded file does not match the canonical CVEfixes structure, so this loader inspects and adapts rather than assuming)
3. Fine-tuning `microsoft/codebert-base` for binary SQL injection classification
4. Running Bandit and Semgrep against the same held-out test set
5. Comparative evaluation (precision, recall, F1, false positive rate, McNemar's test)

**Run every section in order.** Section 0 must run first, every session. If your session disconnects during training, re-running Section 4's training cell automatically resumes from the last saved checkpoint on Drive.

**Before you start:** Runtime → Change runtime type → Hardware accelerator → GPU (T4 is fine on the free tier).

## Section 0 — Environment Setup and Google Drive Mounting

Run this section first, every session.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

PROJECT_ROOT = '/content/drive/MyDrive/sqli_codebert_thesis'
import os
for sub in ['raw_data', 'processed_data', 'checkpoints', 'results', 'tool_outputs']:
    os.makedirs(f'{PROJECT_ROOT}/{sub}', exist_ok=True)
print('Project folders ready under:', PROJECT_ROOT)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Project folders ready under: /content/drive/MyDrive/sqli_codebert_thesis


In [ ]:
!pip install --quiet transformers datasets accelerate evaluate scikit-learn statsmodels bandit semgrep

import torch
print('CUDA available:', torch.cuda.is_available())
print('Device:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU only — check Runtime settings')

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 7.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 134.7/134.7 kB 12.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 70.6/70.6 MB 12.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 223.4/223.4 kB 19.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 782.1/782.1 kB 51.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 193.7/193.7 kB 18.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 119.2/119.2 kB 12.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 103.3/103.3 kB 9.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 90.0/90.0 kB 8.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 65.7/65.7 kB 6.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.5/72.5 kB 7.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 208.0/208.0 kB 11.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

## Section 1 — Configuration

In [ ]:
CONFIG = {
    "project_root": PROJECT_ROOT,
    "raw_data_dir": f"{PROJECT_ROOT}/raw_data",
    "processed_data_dir": f"{PROJECT_ROOT}/processed_data",
    "checkpoint_dir": f"{PROJECT_ROOT}/checkpoints",
    "results_dir": f"{PROJECT_ROOT}/results",
    "tool_outputs_dir": f"{PROJECT_ROOT}/tool_outputs",

    # Dataset
    "cwe_sqli_ids": ["CWE-89", "89", "CWE89"],   # accept multiple formats — files vary
    "target_language_values": ["python", "py"],   # accept multiple formats — files vary
    "sqli_keywords": [  # fallback filter ONLY used if no CWE column exists — see Section 2
        "select ", "insert ", "update ", "delete ", "drop table",
        "execute(", "cursor.execute", "raw(", " query =", "sql =",
    ],
    "max_seq_length": 256,
    "train_ratio": 0.70,
    "val_ratio": 0.15,
    "test_ratio": 0.15,
    "random_seed": 42,
    "min_viable_dataset_size": 40,  # below this, do NOT proceed to training — see Section 2.3

    # Model
    "base_model": "microsoft/codebert-base",
    "num_labels": 2,

    # Training
    "num_epochs": 5,
    "per_device_batch_size": 8,
    "gradient_accumulation_steps": 4,
    "learning_rate": 2e-5,
    "weight_decay": 0.01,
    "eval_strategy": "epoch",
    "save_strategy": "epoch",
}

import json
print(json.dumps(CONFIG, indent=2))

{
  "project_root": "/content/drive/MyDrive/sqli_codebert_thesis",
  "raw_data_dir": "/content/drive/MyDrive/sqli_codebert_thesis/raw_data",
  "processed_data_dir": "/content/drive/MyDrive/sqli_codebert_thesis/processed_data",
  "checkpoint_dir": "/content/drive/MyDrive/sqli_codebert_thesis/checkpoints",
  "results_dir": "/content/drive/MyDrive/sqli_codebert_thesis/results",
  "tool_outputs_dir": "/content/drive/MyDrive/sqli_codebert_thesis/tool_outputs",
  "cwe_sqli_ids": [
    "CWE-89",
    "89",
    "CWE89"
  ],
  "target_language_values": [
    "python",
    "py"
  ],
  "sqli_keywords": [
    "select ",
    "insert ",
    "update ",
    "delete ",
    "drop table",
    "execute(",
    "cursor.execute",
    "raw(",
    " query =",
    "sql ="
  ],
  "max_seq_length": 256,
  "train_ratio": 0.7,
  "val_ratio": 0.15,
  "test_ratio": 0.15,
  "random_seed": 42,
  "min_viable_dataset_size": 40,
  "base_model": "microsoft/codebert-base",
  "num_labels": 2,
  "num_epochs": 5,
  "per_device_

## Section 2 — Dataset Construction (CVEfixes only)

Place your downloaded CVEfixes file at `raw_data/cvefixes/CVEfixes.csv` in Drive before running this section.

**This loader does not assume a fixed schema.** It first prints every column in your file, then detects which of several known column-naming patterns is present (the canonical Bhandari et al. 2021 structure, or the `code`/`safety`/`language` structure your file has shown so far) and adapts. If your file uses yet another naming convention, the printed column list will tell you immediately what to add.

In [ ]:
import pandas as pd
import re
import hashlib

def normalise_code(code: str) -> str:
    """Strip vulnerability-revealing comments; avoids trivial label leakage (Section 3.3.2)."""
    code = re.sub(r'#.*(vulnerable|VULNERABLE|CWE-89|sql injection).*', '', code, flags=re.IGNORECASE)
    return code.strip()

def hash_code(code: str) -> str:
    return hashlib.sha256(normalise_code(code).encode('utf-8')).hexdigest()

In [ ]:
def load_cvefixes(raw_dir: str, config: dict) -> pd.DataFrame:
    """
    Schema-adaptive CVEfixes loader.
    Step 1: read the file and print every column so nothing is assumed silently.
    Step 2: detect the language column and filter to Python.
    Step 3: detect whether CWE-level filtering is possible; if not, fall back to
            a disclosed, keyword-based SQLi heuristic and say so loudly.
    Step 4: detect the code/label columns and normalise into the common record format.
    """
    csv_path = f"{raw_dir}/cvefixes/CVEFixes.csv" # Corrected filename to CVEFixes.csv
    if not os.path.exists(csv_path):
        print(f"[FATAL] {csv_path} not found. Nothing to load. Place the file there first.")
        return pd.DataFrame(columns=["code", "label", "source"])

    df = pd.read_csv(csv_path, low_memory=False)
    print(f"[INFO] Loaded raw file: {len(df)} rows")
    print(f"[INFO] Columns found: {df.columns.tolist()}")

    # --- Step 2: language filter ---
    lang_col = None
    for candidate in ["language", "programming_language", "lang"]:
        if candidate in df.columns:
            lang_col = candidate
            break

    if lang_col is None:
        print("[WARN] No language column found at all — cannot confirm Python-only scope. "
              "STOPPING rather than guessing. Inspect the column list above and tell Claude "
              "which column (if any) indicates language.")
        return pd.DataFrame(columns=["code", "label", "source"])

    print(f"[INFO] Using '{lang_col}' as the language column.")
    print(f"[INFO] Unique values in '{lang_col}': {df[lang_col].dropna().unique().tolist()[:20]}")
    before = len(df)
    df = df[df[lang_col].astype(str).str.lower().isin(config["target_language_values"])]
    print(f"[INFO] Language filter ('python'/'py'): {before} -> {len(df)} rows")

    if len(df) == 0:
        print("[FATAL] Zero rows after language filtering. This file may not contain any "
              "Python entries, or the language values use a format not in target_language_values. "
              "Check the unique values printed above and adjust CONFIG if needed.")
        return pd.DataFrame(columns=["code", "label", "source"])

    # --- Step 3: CWE-level filtering, with disclosed fallback ---
    cwe_col = None
    for candidate in ["cwe_id", "CWE ID", "cwe", "CWE_ID"]:
        if candidate in df.columns:
            cwe_col = candidate
            break

    if cwe_col is not None:
        print(f"[INFO] Using '{cwe_col}' for CWE-89 filtering (exact match).")
        before = len(df)
        df = df[df[cwe_col].astype(str).isin(config["cwe_sqli_ids"])]
        print(f"[INFO] CWE-89 filter: {before} -> {len(df)} rows")
    else:
        print("[WARN] No CWE identifier column found in this file — cannot filter to CWE-89 "
              "directly. Falling back to a KEYWORD-BASED heuristic on the code content itself "
              "(looking for SQL-statement and query-execution patterns). This fallback is a "
              "disclosed methodological limitation, not a silent substitution — it must be "
              "reported as such in Chapter Three if used for the final dataset.")
        code_col_guess = "code" if "code" in df.columns else None
        if code_col_guess is None:
            print("[FATAL] No 'code' column either — cannot apply the keyword fallback. Stopping.")
            return pd.DataFrame(columns=["code", "label", "source"])
        before = len(df)
        pattern = "|".join(re.escape(k) for k in config["sqli_keywords"])
        df = df[df[code_col_guess].astype(str).str.lower().str.contains(pattern, regex=True, na=False)]
        print(f"[INFO] Keyword-based SQLi heuristic filter: {before} -> {len(df)} rows")

    if len(df) == 0:
        print("[FATAL] Zero rows after CWE/keyword filtering. No usable SQLi-relevant Python "
              "samples in this file with current filters.")
        return pd.DataFrame(columns=["code", "label", "source"])

    # --- Step 4: extract code + label ---
    records = []

    if "code_before" in df.columns and "code_after" in df.columns:
        print("[INFO] Using canonical code_before/code_after columns (before=vulnerable, after=fixed).")
        for _, row in df.iterrows():
            if isinstance(row.get("code_before"), str):
                records.append({"code": row["code_before"], "label": 1, "source": "cvefixes"})
            if isinstance(row.get("code_after"), str):
                records.append({"code": row["code_after"], "label": 0, "source": "cvefixes"})

    elif "code" in df.columns and "safety" in df.columns:
        print("[INFO] Using code/safety columns (safety: 'safe' -> 0, 'vulnerable' -> 1).")
        for _, row in df.iterrows():
            if not isinstance(row.get("code"), str):
                continue
            safety_val = str(row.get("safety", "")).strip().lower()
            if safety_val == "safe":
                label = 0
            elif safety_val == "vulnerable":
                label = 1
            else:
                continue  # unknown value — excluded rather than silently defaulted (fixes prior bug)
            records.append({"code": row["code"], "label": label, "source": "cvefixes"})

    else:
        print("[FATAL] Could not find a recognised code/label column pair "
              "(checked code_before/code_after and code/safety). "
              "Columns available: " + str(df.columns.tolist()))
        return pd.DataFrame(columns=["code", "label", "source"])

    out = pd.DataFrame(records)
    print(f"[INFO] Final extracted samples: {len(out)}")
    if len(out) > 0:
        print(f"[INFO] Label distribution:\n{out['label'].value_counts()}")
    return out

In [ ]:
df_cvefixes = load_cvefixes(CONFIG["raw_data_dir"], CONFIG)
print(f"\n=== CVEfixes final count: {len(df_cvefixes)} ===")

[FATAL] /content/drive/MyDrive/sqli_codebert_thesis/raw_data/cvefixes/CVEfixes.csv not found. Nothing to load. Place the file there first.

=== CVEfixes final count: 0 ===


### 2.1 Sanity check — stop here if the dataset is too small

This cell is a deliberate circuit-breaker. It will refuse to let you continue into tokenization/training with a dataset that cannot possibly produce meaningful results (this is exactly what happened previously — do not skip this check).

In [ ]:
combined = df_cvefixes.copy()

n_total = len(combined)
n_classes = combined["label"].nunique() if n_total > 0 else 0

print(f"Total samples: {n_total}")
if n_total > 0:
    print(f"Class distribution:\n{combined['label'].value_counts()}")
    print(f"Number of distinct classes present: {n_classes}")

if n_total < CONFIG["min_viable_dataset_size"]:
    print(f"\n🛑 STOP: only {n_total} samples (< minimum viable {CONFIG['min_viable_dataset_size']}). "
          f"Do NOT proceed to tokenization/training with this — a stratified split would leave "
          f"almost nothing in the test set, producing meaningless metrics (exactly the earlier failure). "
          f"Bring this count back to Claude: you likely need a supplementary Python-native SQLi dataset.")
elif n_classes < 2:
    print(f"\n🛑 STOP: only {n_classes} class present in the data. Both vulnerable and "
          f"non-vulnerable examples are required. Check the label-extraction logic above.")
else:
    print(f"\n✅ Dataset appears viable to proceed. Continue to Section 2.2 (cleaning + split).")

Total samples: 0

🛑 STOP: only 0 samples (< minimum viable 40). Do NOT proceed to tokenization/training with this — a stratified split would leave almost nothing in the test set, producing meaningless metrics (exactly the earlier failure). Bring this count back to Claude: you likely need a supplementary Python-native SQLi dataset.


### 2.2 Cleaning, de-duplication, and stratified split

Only run this once the check above shows ✅.

In [ ]:
combined["code"] = combined["code"].astype(str).apply(normalise_code)
combined = combined[combined["code"].str.len() > 10]

combined["hash"] = combined["code"].apply(hash_code)
before = len(combined)
combined = combined.drop_duplicates(subset="hash").drop(columns="hash").reset_index(drop=True)
print(f"De-duplication: {before} -> {len(combined)} samples")

combined.to_csv(f"{CONFIG['processed_data_dir']}/combined_clean.csv", index=False)
print("Saved combined_clean.csv to Drive.")

De-duplication: 0 -> 0 samples
Saved combined_clean.csv to Drive.


In [ ]:
from sklearn.model_selection import train_test_split

# Stratify by label only (single source now, so no source-stratification needed)
train_df, temp_df = train_test_split(
    combined, test_size=(1 - CONFIG["train_ratio"]),
    stratify=combined["label"], random_state=CONFIG["random_seed"]
)
relative_val = CONFIG["val_ratio"] / (CONFIG["val_ratio"] + CONFIG["test_ratio"])
val_df, test_df = train_test_split(
    temp_df, test_size=(1 - relative_val),
    stratify=temp_df["label"], random_state=CONFIG["random_seed"]
)

for name, part in [("train", train_df), ("val", val_df), ("test", test_df)]:
    part.to_csv(f"{CONFIG['processed_data_dir']}/{name}.csv", index=False)
    print(f"{name}: {len(part)} samples | label distribution:\n{part['label'].value_counts()}\n")

ValueError: With n_samples=0, test_size=0.30000000000000004 and train_size=None, the resulting train set will be empty. Adjust any of the aforementioned parameters.

In [ ]:
ls /content/drive/MyDrive/sqli_codebert_thesis/raw_data/cvefixes/

CVEFixes.csv


## Section 3 — Tokenization and Hugging Face Dataset Creation

In [ ]:
from datasets import Dataset, DatasetDict
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained(CONFIG["base_model"])

def load_split(name):
    df = pd.read_csv(f"{CONFIG['processed_data_dir']}/{name}.csv")
    return Dataset.from_pandas(df[["code", "label"]])

raw_datasets = DatasetDict({
    "train": load_split("train"),
    "validation": load_split("val"),
    "test": load_split("test"),
})

def tokenize_fn(batch):
    return tokenizer(
        batch["code"],
        truncation=True,
        padding="max_length",
        max_length=CONFIG["max_seq_length"],
    )

tokenized_datasets = raw_datasets.map(tokenize_fn, batched=True)
tokenized_datasets = tokenized_datasets.rename_column("label", "labels")
tokenized_datasets.set_format(type="torch", columns=["input_ids", "attention_mask", "labels"])

print(tokenized_datasets)

## Section 4 — Model Definition, Class-Imbalance Handling, and Fine-Tuning

Implements Algorithm 4.1 (Chapter Four) via the Hugging Face `Trainer` API — no hand-written training loop.

In [ ]:
import numpy as np
from transformers import AutoModelForSequenceClassification, TrainingArguments, Trainer
import torch.nn as nn

model = AutoModelForSequenceClassification.from_pretrained(
    CONFIG["base_model"], num_labels=CONFIG["num_labels"]
)

train_labels = np.array(raw_datasets["train"]["label"])
class_counts = np.bincount(train_labels)
class_weights = class_counts.sum() / (len(class_counts) * class_counts)
class_weights_tensor = torch.tensor(class_weights, dtype=torch.float)
print("Class counts:", class_counts, "| Class weights:", class_weights)

In [ ]:
class WeightedTrainer(Trainer):
    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        labels = inputs.pop("labels")
        outputs = model(**inputs)
        logits = outputs.logits
        loss_fct = nn.CrossEntropyLoss(weight=class_weights_tensor.to(logits.device))
        loss = loss_fct(logits, labels)
        return (loss, outputs) if return_outputs else loss

In [ ]:
import evaluate

precision_metric = evaluate.load("precision")
recall_metric = evaluate.load("recall")
f1_metric = evaluate.load("f1")
accuracy_metric = evaluate.load("accuracy")

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    return {
        "accuracy": accuracy_metric.compute(predictions=preds, references=labels)["accuracy"],
        "precision": precision_metric.compute(predictions=preds, references=labels, zero_division=0)["precision"],
        "recall": recall_metric.compute(predictions=preds, references=labels)["recall"],
        "f1": f1_metric.compute(predictions=preds, references=labels)["f1"],
    }

In [ ]:
training_args = TrainingArguments(
    output_dir=CONFIG["checkpoint_dir"],
    num_train_epochs=CONFIG["num_epochs"],
    per_device_train_batch_size=CONFIG["per_device_batch_size"],
    per_device_eval_batch_size=CONFIG["per_device_batch_size"],
    gradient_accumulation_steps=CONFIG["gradient_accumulation_steps"],
    learning_rate=CONFIG["learning_rate"],
    weight_decay=CONFIG["weight_decay"],
    eval_strategy=CONFIG["eval_strategy"],
    save_strategy=CONFIG["save_strategy"],
    save_total_limit=3,
    load_best_model_at_end=True,
    metric_for_best_model="f1",
    greater_is_better=True,
    logging_dir=f"{CONFIG['checkpoint_dir']}/logs",
    logging_steps=50,
    report_to="none",
)

trainer = WeightedTrainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_datasets["train"],
    eval_dataset=tokenized_datasets["validation"],
    compute_metrics=compute_metrics,
)

In [ ]:
import glob

existing_checkpoints = sorted(
    glob.glob(f"{CONFIG['checkpoint_dir']}/checkpoint-*"),
    key=lambda p: int(p.split("-")[-1])
)
resume_path = existing_checkpoints[-1] if existing_checkpoints else None
if resume_path:
    print(f"Resuming training from checkpoint: {resume_path}")
else:
    print("No existing checkpoint found — starting fine-tuning from the base CodeBERT model.")

trainer.train(resume_from_checkpoint=resume_path)

In [ ]:
best_model_dir = f"{CONFIG['checkpoint_dir']}/best_model"
trainer.save_model(best_model_dir)
tokenizer.save_pretrained(best_model_dir)
print(f"Best model saved to: {best_model_dir}")

val_metrics = trainer.evaluate()
print("Validation metrics:", val_metrics)

## Section 5 — CodeBERT Predictions on the Held-Out Test Set

In [ ]:
test_predictions = trainer.predict(tokenized_datasets["test"])
test_logits = test_predictions.predictions
test_preds = np.argmax(test_logits, axis=-1)
test_labels = test_predictions.label_ids

test_results_df = pd.DataFrame({
    "code": raw_datasets["test"]["code"],
    "true_label": test_labels,
    "codebert_pred": test_preds,
})
test_results_df.to_csv(f"{CONFIG['results_dir']}/codebert_test_predictions.csv", index=False)
print(f"CodeBERT test-set predictions saved. Vulnerable predicted: {test_preds.sum()} / {len(test_preds)}")

## Section 6 — Static Analysis Tools on the Same Test Set (Bandit + Semgrep only)

In [ ]:
import subprocess, json as jsonlib

test_files_dir = f"{CONFIG['tool_outputs_dir']}/test_files"
os.makedirs(test_files_dir, exist_ok=True)

test_file_paths = []
for idx, row in test_results_df.iterrows():
    fpath = f"{test_files_dir}/sample_{idx}.py"
    with open(fpath, "w", encoding="utf-8") as f:
        f.write(row["code"])
    test_file_paths.append(fpath)

test_results_df["file_path"] = test_file_paths
print(f"Wrote {len(test_file_paths)} standalone test files for static analysis.")

In [ ]:
def run_bandit(filepath):
    try:
        result = subprocess.run(
            ["bandit", "-f", "json", filepath],
            capture_output=True, text=True, timeout=30
        )
        report = jsonlib.loads(result.stdout) if result.stdout else {"results": []}
        sqli_ids = {"B608"}
        flagged = any(r.get("test_id") in sqli_ids for r in report.get("results", []))
        return int(flagged)
    except Exception as e:
        print(f"[Bandit error on {filepath}]: {e}")
        return 0

test_results_df["bandit_pred"] = test_results_df["file_path"].apply(run_bandit)
print("Bandit flagged:", test_results_df["bandit_pred"].sum(), "/", len(test_results_df))

In [ ]:
SQLI_KEYWORDS = ["sql-injection", "sqli", "sql_injection"]

def run_semgrep(filepath):
    try:
        result = subprocess.run(
            ["semgrep", "--config", "p/python", "--json", "--quiet", filepath],
            capture_output=True, text=True, timeout=60
        )
        report = jsonlib.loads(result.stdout) if result.stdout else {"results": []}
        flagged = any(
            any(kw in r.get("check_id", "").lower() for kw in SQLI_KEYWORDS)
            for r in report.get("results", [])
        )
        return int(flagged)
    except Exception as e:
        print(f"[Semgrep error on {filepath}]: {e}")
        return 0

test_results_df["semgrep_pred"] = test_results_df["file_path"].apply(run_semgrep)
print("Semgrep flagged:", test_results_df["semgrep_pred"].sum(), "/", len(test_results_df))

In [ ]:
test_results_df.to_csv(f"{CONFIG['results_dir']}/consolidated_predictions.csv", index=False)
print("Saved consolidated_predictions.csv")
test_results_df.head()

## Section 7 — Comparative Evaluation

In [ ]:
from sklearn.metrics import precision_score, recall_score, f1_score, accuracy_score, confusion_matrix

approaches = {
    "CodeBERT (developed model)": "codebert_pred",
    "Bandit": "bandit_pred",
    "Semgrep": "semgrep_pred",
}

y_true = test_results_df["true_label"]
summary_rows = []

for name, col in approaches.items():
    y_pred = test_results_df[col]
    tn, fp, fn, tp = confusion_matrix(y_true, y_pred, labels=[0, 1]).ravel()
    fpr = fp / (fp + tn) if (fp + tn) > 0 else 0.0
    summary_rows.append({
        "Approach": name,
        "Precision": round(precision_score(y_true, y_pred, zero_division=0), 4),
        "Recall": round(recall_score(y_true, y_pred, zero_division=0), 4),
        "F1": round(f1_score(y_true, y_pred, zero_division=0), 4),
        "Accuracy": round(accuracy_score(y_true, y_pred), 4),
        "False Positive Rate": round(fpr, 4),
        "TP": tp, "TN": tn, "FP": fp, "FN": fn,
    })

summary_df = pd.DataFrame(summary_rows)
summary_df.to_csv(f"{CONFIG['results_dir']}/comparative_metrics_summary.csv", index=False)
summary_df

In [ ]:
from statsmodels.stats.contingency_tables import mcnemar

def mcnemar_pairwise(y_true, pred_a, pred_b, label_a, label_b):
    correct_a = (pred_a == y_true)
    correct_b = (pred_b == y_true)
    n01 = ((~correct_a) & (correct_b)).sum()
    n10 = ((correct_a) & (~correct_b)).sum()
    table = [[0, n01], [n10, 0]]
    result = mcnemar(table, exact=(n01 + n10 < 25))
    print(f"{label_a} vs {label_b}: statistic={result.statistic:.4f}, p-value={result.pvalue:.4f}"
          f"  (n01={n01}, n10={n10})")
    return result.pvalue

print("McNemar's test — CodeBERT vs each static analysis tool:\n")
for name, col in approaches.items():
    if name == "CodeBERT (developed model)":
        continue
    mcnemar_pairwise(
        y_true,
        test_results_df["codebert_pred"],
        test_results_df[col],
        "CodeBERT", name,
    )

## Section 8 — Wrap-Up

`results/comparative_metrics_summary.csv` and `results/consolidated_predictions.csv` on Drive now contain everything Chapter Five needs. If the Section 2.1 sanity check passed with a genuinely adequate sample count, these numbers are real and usable — unlike the earlier 3-sample run.